# Osonye Onyemazuwa — Week 3: Metric Calculation & Data Modeling
### Day 15: Calculate Total Spend per channel/campaign/day (SQL aggregation)

**Issue #59:** Calculate Total Spend per channel/campaign/day.

Week 2 already built separate views for each dimension
(`vw_spend_by_channel`, `vw_spend_by_campaign`, `vw_spend_by_day`).
Rather than duplicate those, this uses `GROUPING SETS` to produce a
single query that returns channel totals, campaign totals, day totals,
**and** the grand total all in one result set -- this is what
"Total Spend per channel/campaign/day" actually means as one KPI
deliverable, not three separate ones.


In [ ]:
import os
import urllib.parse
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
DB_USER = os.environ.get("DB_USER")
DB_PASSWORD = urllib.parse.quote_plus(os.environ.get("DB_PASSWORD"))
DB_HOST = os.environ.get("DB_HOST")
DB_PORT = os.environ.get("DB_PORT")
DB_NAME = os.environ.get("DB_NAME")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("Connected.")


## Total Spend by channel (reusing Week 2's view)

In [ ]:
query = text("SELECT channel, total_spend FROM vw_spend_by_channel ORDER BY total_spend DESC")
with engine.connect() as conn:
    for row in conn.execute(query):
        print(row)


Expected: affiliate **\$6,454.96** down to social **\$4,176.33** -- same numbers
as Week 2 Day 11, confirming the view is still consistent.


## Total Spend by campaign (top 10, reusing Week 2's view)

In [ ]:
query = text("""
SELECT campaign_id, channel, total_spend
FROM vw_spend_by_campaign
ORDER BY total_spend DESC
LIMIT 10
""")
with engine.connect() as conn:
    for row in conn.execute(query):
        print(row)


## Total Spend by day (summary stats, reusing Week 2's view)

In [ ]:
query = text("""
SELECT
    COUNT(*) AS total_days,
    MIN(total_spend) AS min_daily_spend,
    ROUND(AVG(total_spend), 2) AS avg_daily_spend,
    MAX(total_spend) AS max_daily_spend
FROM vw_spend_by_day
""")
with engine.connect() as conn:
    result = conn.execute(query).fetchone()
print("Total days:", result[0])
print("Min daily spend:", result[1])
print("Avg daily spend:", result[2])
print("Max daily spend:", result[3])


Expected: 967 total days, matching Week 1/2\'s earlier findings (**\$10.00**
min, **~\$27.79** avg, **\$78.36** max daily spend).


## Combined Total Spend across all three dimensions in one query

In [ ]:
query = text("""
SELECT
    channel,
    campaign_id,
    date,
    ROUND(SUM(ad_spend), 2) AS total_spend
FROM ad_spend
GROUP BY GROUPING SETS (
    (channel),
    (campaign_id),
    (date),
    ()
)
ORDER BY total_spend DESC NULLS LAST
LIMIT 15
""")
with engine.connect() as conn:
    for row in conn.execute(query):
        print(row)


`GROUPING SETS` computes multiple GROUP BY levels in a single scan of
the table -- channel totals, campaign totals, day totals, and the grand
total, all in one query rather than four separate ones. Rows where
`channel`/`campaign_id`/`date` show as NULL indicate that row is a
subtotal from a *different* grouping level (e.g. a row with only
`campaign_id` filled in and `channel`/`date` NULL is a campaign-level
total, not a row with missing data).


In [ ]:
query = text("""
SELECT ROUND(SUM(ad_spend), 2) AS grand_total
FROM ad_spend
GROUP BY GROUPING SETS (())
""")
with engine.connect() as conn:
    grouping_sets_total = conn.execute(query).scalar()

query2 = text("SELECT ROUND(SUM(ad_spend), 2) FROM ad_spend")
with engine.connect() as conn:
    raw_total = conn.execute(query2).scalar()

print("GROUPING SETS grand total:", grouping_sets_total)
print("Raw table total:          ", raw_total)
print("Match:", grouping_sets_total == raw_total)


Expected: both equal $26,876.24, `Match: True`. This confirms the
GROUPING SETS query isn't silently dropping or double-counting any
rows across its four grouping levels.


## Day 15 notes

- Reused Week 2's three views (`vw_spend_by_channel`,
  `vw_spend_by_campaign`, `vw_spend_by_day`) rather than duplicating
  their logic -- all three still return identical numbers to Week 2,
  confirming the underlying data hasn't drifted
- Added a `GROUPING SETS` query to genuinely fulfill "Total Spend per
  channel/campaign/day" as one combined deliverable, computing all
  three grouping levels plus the grand total in a single table scan
- Verified the GROUPING SETS grand total ($26,876.24) matches the raw
  `ad_spend` table total exactly -- no rows lost or double-counted
  across the four grouping levels
